# Variable Scope in Python — In Depth (including OOP / Classes)

This notebook covers:
1. What is variable scope?
2. The LEGB rule (Local, Enclosing, Global, Built-in)
3. `global` and `nonlocal` keywords
4. Variable scope inside **classes**
5. Class variables vs Instance variables
6. Scope inside methods (`self`, `cls`)
7. Name mangling (`__var`)
8. `staticmethod` and `classmethod` scope behavior
9. Common pitfalls & interview-style gotchas


## 1. What is Variable Scope?

**Scope** = the region of code where a variable name is *visible / accessible*.

Python resolves a variable name using the **LEGB rule**:

| Level | Meaning |
|---|---|
| **L** – Local | Inside the current function |
| **E** – Enclosing | Inside any enclosing function (closures) |
| **G** – Global | Top level of the module/script |
| **B** – Built-in | Python's built-in names (`len`, `print`, etc.) |

Python searches in this order: **Local → Enclosing → Global → Built-in**, and stops at the first match.


In [18]:
x = "global x"

def outer():
    x = "enclosing x"
    def inner():
        x = "local x"
        print("Inner sees:", x)      # Local
    inner()
    print("Outer sees:", x)          # Enclosing

outer()
print("Module sees:", x)             # Global


Inner sees: local x
Outer sees: enclosing x
Module sees: global x


### Built-in scope example

In [19]:
# 'len' is a built-in, not defined anywhere by us
print(len([1, 2, 3]))   # B - Built-in scope resolves this


3


## 2. `global` and `nonlocal` keywords

- By default, assigning to a variable inside a function creates a **new local variable** — it does NOT modify an outer one.
- `global` lets you modify a **module-level (global)** variable from inside a function.
- `nonlocal` lets you modify a variable in the **nearest enclosing (non-global) scope**.


In [20]:
counter = 0

def increment_wrong():
    counter = counter + 1   # UnboundLocalError! Python treats counter as local here
    return counter

try:
    increment_wrong()
except UnboundLocalError as e:
    print("Error:", e)


Error: cannot access local variable 'counter' where it is not associated with a value


In [21]:
counter = 0

def increment_right():
    global counter
    counter += 1
    return counter

print(increment_right())
print(counter)
print(increment_right())
print(increment_right())
print("Module-level counter is now:", counter)


1
1
2
3
Module-level counter is now: 3


In [7]:
def outer():
    count = 0
    def inner():
        nonlocal count
        count += 1
        return count
    print(inner())
    print(count)
    print(inner())
    print(count)
    print("Outer's count after calls:", count)

outer()


1
1
2
2
Outer's count after calls: 2


In [22]:
def outer():
    def inner():
        x = 1
        nonlocal x
    inner()

outer()

SyntaxError: name 'x' is assigned to before nonlocal declaration (1163812956.py, line 4)

## 3. Variable Scope and Classes — the important part

A **class body** itself has its own local namespace while it's being executed (once, at class-definition time).
But this is where scoping gets tricky for beginners:

> **A class body's namespace is NOT treated as an enclosing scope for methods defined inside it.**

That means methods do **not** automatically see class-body variables the way `inner()` sees `outer()`'s variables. You must access them via `self.` or `ClassName.`.


In [8]:
class Demo:
    class_var = "I am a class variable"   # lives in the class's own namespace

    def show(self):
        # print(class_var)   # NameError! class body is NOT an enclosing scope for methods
        print(self.class_var)        # correct: via instance (falls back to class)
        print(Demo.class_var)        # correct: via class name directly

d = Demo()
d.show()


I am a class variable
I am a class variable


In [24]:
# Proving the NameError if we try direct access
class Demo2:
    class_var = "hello"
    def broken(self):
        return class_var   # looked up via LEGB -> Local, Enclosing(none), Global, Built-in
                            # class body is skipped entirely!

try:
    Demo2().broken()
except NameError as e:
    print("NameError:", e)


NameError: name 'class_var' is not defined


### Why does this happen?

LEGB for a method inside a class looks like:

```
Local (method body) -> Enclosing (skips class body!) -> Global (module) -> Built-in
```

The class body is executed **once** to build the class object, and its namespace becomes the class's `__dict__`. It is *not* wired up as a closure/enclosing scope for the methods defined within it. This is a deliberate Python design choice.


## 4. Class Variables vs Instance Variables

| | Class variable | Instance variable |
|---|---|---|
| Defined | Directly in class body | Usually in `__init__` via `self.x = ...` |
| Storage | `ClassName.__dict__` | `instance.__dict__` |
| Shared? | **Shared** across all instances | Unique per instance |
| Access | `ClassName.var` or `self.var` (read) | `self.var` |


In [10]:
class Counter:
    total_created = 0          # class variable (shared)

    def __init__(self, name):
        self.name = name       # instance variable (unique)
        Counter.total_created += 1   # modify via class name, not self!

c1 = Counter("A")
c2 = Counter("B")
c3 = Counter("C")

print(c1.name, c2.name, c3.name)
print("Total created:", Counter.total_created)
print(c1.total_created, c2.total_created)  # instances can READ the class var too


A B C
Total created: 3
3 3


In [11]:
# GOTCHA: self.var = ... creates a NEW instance variable, it does NOT modify the class variable!

class Counter2:
    total = 0
    def bump_wrong(self):
        self.total += 1     # reads class var (0), then creates an INSTANCE var 'total' = 1

a = Counter2()
b = Counter2()
a.bump_wrong()
a.bump_wrong()

print("a.total:", a.total)          # 2 (instance variable, shadowing class var)
print("b.total:", b.total)          # 0 (still reads class variable, untouched)
print("Counter2.total:", Counter2.total)  # 0 (class variable never changed)


a.total: 2
b.total: 0
Counter2.total: 0


## 5. Scope inside methods — `self` and `cls`

- `self` is just a regular parameter — it exists in the method's **local scope**, like any other argument. Python doesn't do this automatically by magic; it's convention + how `instance.method()` passes the instance as the first argument.
- `cls` plays the same local-scope role inside a `@classmethod`, representing the class itself.


In [12]:
class Person:
    species = "Homo sapiens"

    def __init__(self, name):
        self.name = name        # self is local to __init__

    def greet(self):
        # self is local to greet(); name is looked up as self.name (attribute access)
        greeting = f"Hi, I'm {self.name}"   # 'greeting' is local to greet()
        return greeting

    @classmethod
    def get_species(cls):
        return cls.species      # cls is local to this method

    @staticmethod
    def generic_greeting():
        # NOTE: no self/cls here at all -> cannot access instance or class attributes directly
        return "Hello there!"

p = Person("Ravi")
print(p.greet())
print(Person.get_species())
print(Person.generic_greeting())
# print(greeting)   # NameError -> 'greeting' was local to greet(), doesn't exist here


Hi, I'm Ravi
Homo sapiens
Hello there!


## 6. Name Mangling — `__variable` (double leading underscore)

Inside a class, a name like `__var` (two leading underscores, at most one trailing) gets **automatically renamed** by Python to `_ClassName__var`. This isn't true privacy/scoping like other languages have — it's a convention to avoid accidental name clashes in subclasses.


In [13]:
class Base:
    def __init__(self):
        self.__secret = "base secret"    # becomes self._Base__secret

    def reveal(self):
        return self.__secret             # works fine inside the class

b = Base()
print(b.reveal())
print(b._Base__secret)     # accessible via mangled name (not true privacy)

try:
    print(b.__secret)      # AttributeError! '__secret' doesn't exist as-is
except AttributeError as e:
    print("AttributeError:", e)


base secret
base secret
AttributeError: 'Base' object has no attribute '__secret'


In [14]:
# Name mangling avoiding clashes in inheritance
class Base2:
    def __init__(self):
        self.__x = "base"     # -> _Base2__x
    def show_base(self):
        return self.__x

class Child2(Base2):
    def __init__(self):
        super().__init__()
        self.__x = "child"    # -> _Child2__x  (DIFFERENT attribute, no clash!)
    def show_child(self):
        return self.__x

c = Child2()
print(c.show_base())    # 'base'  -> uses _Base2__x
print(c.show_child())   # 'child' -> uses _Child2__x
print(vars(c))           # shows both mangled attributes co-existing


base
child
{'_Base2__x': 'base', '_Child2__x': 'child'}


## 7. Closures inside classes (methods returning functions)

A method *can* create a closure over local variables, and that closure follows normal LEGB rules — it's the **class body** specifically that's excluded from enclosing scope, not methods themselves.


In [15]:
class Multiplier:
    def make_multiplier(self, factor):
        def multiply(x):
            return x * factor   # 'factor' is captured from enclosing (make_multiplier) scope
        return multiply

m = Multiplier()
times3 = m.make_multiplier(3)
print(times3(10))   # 30


30


## 8. Comprehensions have their own local scope (Python 3)

List/dict/set comprehensions and generator expressions create their **own local scope**, separate from the surrounding function/class — this trips people up inside class bodies especially.


In [16]:
class Squares:
    n = 5
    # This WORKS: comprehension's iterable expression (range(n)) is evaluated in class scope,
    # but the loop variable and body run in the comprehension's own scope.
    values = [i * i for i in range(n)]

print(Squares.values)


[0, 1, 4, 9, 16]


In [17]:
class Broken:
    n = 5
    multiplier = 2
    try:
        # This FAILS: inside the comprehension body, class-body names like 'multiplier'
        # are not visible (same rule as methods not seeing class-body names directly)
        values = [i * multiplier for i in range(n)]
    except NameError as e:
        values = f"NameError: {e}"

print(Broken.values)


NameError: name 'multiplier' is not defined


## 9. Summary Table

| Scope type | Where it lives | Visible to nested functions/methods? |
|---|---|---|
| Local | Inside a function/method | N/A (it IS the local scope) |
| Enclosing | Outer function (closures) | Yes, automatically |
| Global | Module top level | Yes (read); needs `global` to write |
| Built-in | Python itself | Yes, everywhere |
| **Class body** | Class definition namespace | **No** — must use `self.` / `ClassName.` |
| Instance (`self.x`) | `instance.__dict__` | Only via `self` |
| Class variable | `ClassName.__dict__` | Via `self.` (read, falls back) or `ClassName.` (read/write) |

### Key takeaways
1. LEGB governs *function-level* scoping; the **class body is skipped** as an enclosing scope for methods.
2. Use `self.x` to read/write instance attributes, `ClassName.x` to write class attributes reliably.
3. `self.x = value` inside a method **always creates/updates an instance attribute**, even if a class attribute with the same name exists — it shadows it, it does not mutate it.
4. `__name` mangling is about avoiding subclass collisions, not real privacy.
5. Comprehensions get their own scope, even inside a class body.
